# 🚖 下車地址推薦系統

**架構：兩階段推薦**
- Stage 1 (Candidate Generation)：從 `address_v2_suggestion` 撈出用戶歷史下車地址候選集
- Stage 2 (Ranking)：用 **LightGBM LambdaRank** 對候選集排序，輸出 Top-K 推薦

**特徵（12 個）：** 原始頻率 × 6 + log1p 轉換版 × 6

| 特徵 | 說明 |
|---|---|
| `user_end_freq` | 用戶歷史去過該地址幾次（最強信號）|
| `global_end_freq` | 全局熱門度（cold-start 用）|
| `hour_end_freq` | 同時段去該地址的條件頻率 |
| `holiday_end_freq` | 同假日/平日狀態的條件頻率 |
| `dow_end_freq` | 同星期幾的條件頻率 |
| `start_end_freq` | 從同一上車區域出發去該地址的頻率 |

**評估指標：** Recall@K、MRR@K、NDCG@K（K = 1, 3, 5）

---
### 使用說明
1. 把 `address_v2_training_data.parquet` 和 `address_v2_suggestion.parquet` 上傳到 Google Drive
2. 在 **⚙️ 設定** 那格填入正確的檔案路徑
3. 依序執行所有 Cell（`執行階段 → 全部執行`）

## 0｜安裝套件

In [1]:
# lightgbm --upgrade 確保版本支援 GPU
!pip install lightgbm --upgrade tqdm pyarrow -q

# 把同學的工具檔案上傳到 Colab 工作目錄（若已上傳可跳過）
# 需要: data_loader.py / evaluate.py / read_parquet.py
# 放在 /content/ 底下即可，或直接上傳到左側檔案面板


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.1 MB/s eta 0:00:00


## 1｜掛載 Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive 已掛載')

Mounted at /content/drive
✓ Google Drive 已掛載


## ⚙️ 設定（請修改這格）

In [3]:
from pathlib import Path
import sys

# ── Google Drive 路徑 ────────────────────────────────────────────────
DRIVE_ROOT  = Path('/content/drive/MyDrive')
DATA_FOLDER = DRIVE_ROOT / 'LineGO_data'   # ← 改成你放資料的資料夾
OUTPUT_DIR  = DRIVE_ROOT / 'LineGO_checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 把同學的工具檔加進 sys.path（假設放在 /content/）───────────────────
# 若放在其他路徑，修改下面這行
UTILS_DIR = Path('/content')
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

# ── 讓 read_parquet.py 找得到資料（它用 Path(__file__).parent / '1.下車地址推薦'）
# 把兩個 parquet 放在 /content/1.下車地址推薦/ 或修改 read_parquet.py 的 BASE_DIR
import os
DATA_LINK = Path('/content/1.下車地址推薦')
if not DATA_LINK.exists():
    DATA_LINK.mkdir(parents=True, exist_ok=True)
    # 從 Drive 建 symlink（不複製，省空間）
    for fname in ['address_v2_training_data.parquet', 'address_v2_suggestion.parquet']:
        src = DATA_FOLDER / fname
        dst = DATA_LINK / fname
        if src.exists() and not dst.exists():
            os.symlink(src, dst)

# ── 模型超參數 ────────────────────────────────────────────────────────
TOP_K_LIST = [1, 3, 5]
NEG_RATIO  = 10      # 每個正樣本配幾個負樣本
N_ROUNDS   = 500    # LightGBM 最大迭代數
EARLY_STOP = 30     # val NDCG 連續幾輪不進步就停止
SEED       = 42

FEAT_COLS = ['user_end_freq', 'global_end_freq', 'hour_end_freq',
             'holiday_end_freq', 'dow_end_freq', 'start_end_freq']
ALL_FEAT  = FEAT_COLS + [f'log_{c}' for c in FEAT_COLS]

print('✓ 設定完成')


✓ 設定完成


## 2｜Import

In [7]:
import gc, json, pickle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
from tqdm.notebook import tqdm

# 同學的工具模組
from data_loader import load_split      # 時序切分，全量用戶
from evaluate import evaluate as eval_topk, EvalResult  # row-level 評估

print('✓ 所有套件載入完成')


✓ 所有套件載入完成


## 2.5｜GPU 偵測

In [8]:
import subprocess

def detect_gpu():
    """偵測是否有可用的 GPU，並設定 LightGBM 訓練裝置。"""
    global USE_GPU, LGB_DEVICE_PARAMS

    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
             '--format=csv,noheader'],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0 and result.stdout.strip():
            gpu_info = result.stdout.strip().split('\n')
            print('🟢 GPU 可用！')
            for i, info in enumerate(gpu_info):
                name, mem, driver = [x.strip() for x in info.split(',')]
                print(f'   GPU {i}: {name} | 顯存: {mem} | Driver: {driver}')
            USE_GPU = True
            LGB_DEVICE_PARAMS = {
                'device':          'gpu',
                'gpu_platform_id': 0,
                'gpu_device_id':   0,
            }
            print('   ✓ LightGBM 將使用 GPU 訓練')
        else:
            raise RuntimeError('nvidia-smi 無回應')
    except Exception as e:
        print(f'🔴 未偵測到 GPU（{e}）')
        print('   → 使用 CPU 訓練（可至 執行階段 → 變更執行階段類型 → T4 GPU 切換）')
        USE_GPU = False
        LGB_DEVICE_PARAMS = {}  # 空 dict = CPU 模式，不改任何參數

    # 額外驗證：讓 LightGBM 實際試跑一個最小 GPU dataset
    if USE_GPU:
        try:
            import lightgbm as lgb, numpy as np
            _X = np.random.rand(100, 4).astype(np.float32)
            _y = np.random.randint(0, 2, 100).astype(np.float32)
            _ds = lgb.Dataset(_X, label=_y)
            lgb.train(
                {'objective': 'binary', 'verbosity': -1, **LGB_DEVICE_PARAMS},
                _ds, num_boost_round=3
            )
            print('   ✓ LightGBM GPU 驗證通過')
        except Exception as e:
            print(f'   ⚠️  LightGBM GPU 驗證失敗：{e}')
            print('   → 自動退回 CPU 模式')
            USE_GPU = False
            LGB_DEVICE_PARAMS = {}

USE_GPU = False
LGB_DEVICE_PARAMS = {}
detect_gpu()

🟢 GPU 可用！
   GPU 0: Tesla T4 | 顯存: 15360 MiB | Driver: 580.82.07
   ✓ LightGBM 將使用 GPU 訓練
   ✓ LightGBM GPU 驗證通過


## 3｜讀取資料

In [9]:
def load_data():
    """
    使用 data_loader.load_split() 做時序切分（全量用戶，不過濾 MIN_TRIPS）。
    切分邏輯: 全量行程依 created_at 排序 → train 75% / val 10% / test 15%
    """
    print('[1/5] 讀取資料（使用 data_loader）...')
    with tqdm(total=3, desc='  載入 splits', unit='split') as pbar:
        pbar.set_postfix_str('train')
        df_train_raw = load_split('train')
        pbar.update(1)

        pbar.set_postfix_str('val')
        df_val_raw = load_split('val')
        pbar.update(1)

        pbar.set_postfix_str('test')
        df_test_raw = load_split('test')
        pbar.update(1)

    # 讀 suggestion 表（地址對照用）
    table   = pq.read_table(DATA_LINK / 'address_v2_suggestion.parquet')
    df_sugg = pd.DataFrame({c: table.column(c).to_pylist() for c in table.column_names})
    del table; gc.collect()

    def _info(name, df):
        t0 = df['created_at'].min().strftime('%Y/%m/%d')
        t1 = df['created_at'].max().strftime('%Y/%m/%d')
        print(f'  {name:<8} {len(df):>7,} 筆  '
              f'{df["uid_hash"].nunique():>6,} 用戶  {t0} ~ {t1}')

    print()
    _info('train',  df_train_raw)
    _info('val',    df_val_raw)
    _info('test',   df_test_raw)
    print(f'  suggestion  {len(df_sugg):>7,} 筆')
    return df_train_raw, df_val_raw, df_test_raw, df_sugg

df_train_raw, df_val_raw, df_test_raw, df_sugg = load_data()


[1/5] 讀取資料（使用 data_loader）...


  載入 splits:   0%|          | 0/3 [00:00<?, ?split/s]


  train    750,000 筆  273,683 用戶  2026/01/01 ~ 2026/04/14
  val      100,000 筆  65,461 用戶  2026/04/14 ~ 2026/04/28
  test     150,000 筆  89,435 用戶  2026/04/28 ~ 2026/05/17
  suggestion  2,493,639 筆


## 4｜預計算頻率查找表

In [10]:
def build_lookup_tables(df_train_raw: pd.DataFrame,
                         df_sugg: pd.DataFrame) -> dict:
    """
    從 train 行程計算所有頻率查找表。
    新增 SuggestionFusion 的 9 個信號：
      uc3: user × (hour, holiday, dow) → end
      uc2: user × (hour, holiday)      → end
      ua : user                        → end
      us : user × start                → end
      sc : (start, hour, holiday)      → end
      sa : start                       → end
      gc : global                      → end
      poi: user × address (POI)        → end
      sugg_user: suggestion 表中 user 去過該地址幾次 (新增)
    """
    import math
    from collections import defaultdict

    print('[2/5] 建立頻率查找表 (僅用 train)...')

    # suggestion 表：end_latlng → pin 座標、address
    latlng_to_pin  = dict(zip(df_sugg['end_latlng'], df_sugg['end_latlng_pin']))
    latlng_to_addr = dict(zip(df_sugg['end_latlng'], df_sugg['end_address']))

    # suggestion 表：user → {end_latlng: count}（出現幾次）
    sugg_user_end = defaultdict(lambda: defaultdict(int))
    with tqdm(total=len(df_sugg), desc='  sugg user count', leave=False) as pbar:
        for uid, end in zip(df_sugg['uid_hash'].values, df_sugg['end_latlng'].values):
            sugg_user_end[uid][end] += 1
            pbar.update(1)
    sugg_user_end = {k: dict(v) for k, v in sugg_user_end.items()}

    # train 行程：9 個信號
    uc3 = defaultdict(lambda: defaultdict(float))
    uc2 = defaultdict(lambda: defaultdict(float))
    ua  = defaultdict(lambda: defaultdict(float))
    us  = defaultdict(lambda: defaultdict(float))
    sc  = defaultdict(lambda: defaultdict(int))
    sa  = defaultdict(lambda: defaultdict(int))
    gc  = defaultdict(int)
    poi = defaultdict(float)   # key: (uid, address)

    cols = df_train_raw[['uid_hash','start_latlng','end_latlng',
                          'hour_type','is_holiday','dayofweek']].values
    with tqdm(cols, desc='  train signals', leave=False) as pbar:
        for uid, start, end, hour, holiday, dow in pbar:
            uc3[(uid, hour, holiday, dow)][end] += 1
            uc2[(uid, hour, holiday)][end]      += 1
            ua[uid][end]                        += 1
            us[(uid, start)][end]               += 1
            sc[(start, hour, holiday)][end]     += 1
            sa[start][end]                      += 1
            gc[end]                             += 1
            addr = latlng_to_addr.get(end)
            if addr:
                poi[(uid, addr)] += 1

    lookups = {
        'uc3': {k: dict(v) for k, v in uc3.items()},
        'uc2': {k: dict(v) for k, v in uc2.items()},
        'ua':  {k: dict(v) for k, v in ua.items()},
        'us':  {k: dict(v) for k, v in us.items()},
        'sc':  {k: dict(v) for k, v in sc.items()},
        'sa':  {k: dict(v) for k, v in sa.items()},
        'gc':  dict(gc),
        'poi': dict(poi),
        'sugg_user_end': sugg_user_end,
        'latlng_to_pin': latlng_to_pin,
        'latlng_to_addr': latlng_to_addr,
    }

    sizes = {k: len(lookups[k]) for k in
             ['uc3','uc2','ua','us','sc','sa','gc','poi','sugg_user_end']}
    print('  ✓ lookup sizes: ' + ', '.join(f'{k}={v:,}' for k,v in sizes.items()))
    return lookups

lookups = build_lookup_tables(df_train_raw, df_sugg)


[2/5] 建立頻率查找表 (僅用 train)...


  sugg user count:   0%|          | 0/2493639 [00:00<?, ?it/s]

  train signals:   0%|          | 0/750000 [00:00<?, ?it/s]

  ✓ lookup sizes: uc3=626,279, uc2=511,784, ua=273,683, us=474,743, sc=32,392, sa=5,919, gc=66,988, poi=533,339, sugg_user_end=659,806


## 5｜建立訓練樣本

In [11]:
import math as _math

FEAT_NAMES_RAW = [
    'log_uc3', 'log_uc2', 'log_ua', 'log_us',
    'log_sc',  'log_sa',  'log_gc', 'log_poi', 'dist'
]
ALL_FEAT = FEAT_NAMES_RAW  # 9 個特徵


def _get_feat(lookups, uid, end, hour, holiday, dow, start):
    f_uc3 = lookups['uc3'].get((uid, hour, holiday, dow), {}).get(end, 0)
    f_uc2 = lookups['uc2'].get((uid, hour, holiday), {}).get(end, 0)
    f_ua  = lookups['ua'].get(uid, {}).get(end, 0)
    f_us  = lookups['us'].get((uid, start), {}).get(end, 0)
    f_sc  = lookups['sc'].get((start, hour, holiday), {}).get(end, 0)
    f_sa  = lookups['sa'].get(start, {}).get(end, 0)
    f_gc  = lookups['gc'].get(end, 0)
    addr  = lookups['latlng_to_addr'].get(end)
    f_poi = lookups['poi'].get((uid, addr), 0) if addr else 0

    dist = 0.0
    pin = lookups['latlng_to_pin'].get(end)
    try:
        slat, slng = start.split(',')
        elat, elng = pin.split(',')
        dlat = (float(elat) - float(slat)) * 111.0
        dlng = (float(elng) - float(slng)) * 101.0
        dist = 1.0 / (1.0 + _math.sqrt(dlat*dlat + dlng*dlng))
    except:
        pass

    return [
        _math.log1p(f_uc3),
        _math.log1p(f_uc2),
        _math.log1p(f_ua),
        _math.log1p(f_us),
        _math.log1p(f_sc),
        _math.log1p(f_sa),
        _math.log1p(f_gc),
        _math.log1p(f_poi),
        dist,
    ]


def _df_to_samples(df_raw, df_sugg, lookups, split_name):
    sugg_dict = (df_sugg.groupby('uid_hash')['end_latlng']
                        .apply(lambda g: g.drop_duplicates().tolist())
                        .to_dict())
    rows = []
    groups = list(df_raw.groupby('uid_hash', sort=False))
    with tqdm(groups, desc=f'  {split_name}', unit='user', leave=False) as pbar:
        for uid, grp in pbar:
            cands = sugg_dict.get(uid, [])
            if not cands: continue
            cand_set = set(cands)
            for _, trip in grp.iterrows():
                true_end = trip['end_latlng']
                if true_end not in cand_set: continue
                neg_cands = [c for c in cands if c != true_end]
                n_neg = min(len(neg_cands), NEG_RATIO)
                sel   = (np.random.choice(len(neg_cands), n_neg, replace=False)
                         if n_neg > 0 else [])
                h, hol, dow, start = (trip['hour_type'], trip['is_holiday'],
                                      trip['dayofweek'], trip['start_latlng'])
                rows.append((uid, 1,
                             _get_feat(lookups, uid, true_end, h, hol, dow, start)))
                for i in sel:
                    rows.append((uid, 0,
                                 _get_feat(lookups, uid, neg_cands[i], h, hol, dow, start)))
    return rows


def to_df(rows):
    feats = np.array([r[2] for r in rows], dtype=np.float32)
    df = pd.DataFrame(feats, columns=ALL_FEAT)
    df.insert(0, 'label',    [r[1] for r in rows])
    df.insert(0, 'uid_hash', [r[0] for r in rows])
    return df


def build_samples(df_train_raw, df_val_raw, df_test_raw, df_sugg, lookups):
    print('[3/5] 建立訓練樣本...')
    np.random.seed(SEED)
    rows_tr   = _df_to_samples(df_train_raw, df_sugg, lookups, 'train')
    rows_val  = _df_to_samples(df_val_raw,   df_sugg, lookups, 'val')
    rows_test = _df_to_samples(df_test_raw,  df_sugg, lookups, 'test')

    df_train = to_df(rows_tr)
    df_val   = to_df(rows_val)
    df_test  = to_df(rows_test)

    for name, df in [('train', df_train), ('val', df_val), ('test', df_test)]:
        print(f'  ✓ {name:<6} {len(df):>7,} 樣本  pos_rate={df["label"].mean():.3f}')

    df_train.to_parquet(OUTPUT_DIR / 'samples_train.parquet', index=False)
    df_val.to_parquet(OUTPUT_DIR   / 'samples_val.parquet',   index=False)
    df_test.to_parquet(OUTPUT_DIR  / 'samples_test.parquet',  index=False)
    print('  ✓ 樣本已存至 Drive')
    return df_train, df_val, df_test


df_train, df_val, df_test = build_samples(
    df_train_raw, df_val_raw, df_test_raw, df_sugg, lookups
)

[3/5] 建立訓練樣本...


  train:   0%|          | 0/273683 [00:00<?, ?user/s]

  val:   0%|          | 0/65461 [00:00<?, ?user/s]

  test:   0%|          | 0/89435 [00:00<?, ?user/s]

  ✓ train  5,627,381 樣本  pos_rate=0.133
  ✓ val    744,663 樣本  pos_rate=0.134
  ✓ test   1,106,374 樣本  pos_rate=0.136
  ✓ 樣本已存至 Drive


## 6｜訓練 LightGBM LambdaRank

In [12]:
def _tqdm_lgb_callback(pbar):
    prev = [0]
    def callback(env):
        n = env.iteration + 1
        pbar.update(n - prev[0])
        prev[0] = n
        if env.evaluation_result_list:
            val_metric = env.evaluation_result_list[-3]
            pbar.set_postfix_str(f'val NDCG@1={val_metric[2]:.4f}')
    callback.order = 10
    return callback


def train_model(df_train: pd.DataFrame, df_val: pd.DataFrame) -> lgb.Booster:
    print('[4/5] 訓練 LightGBM LambdaRank...')
    print(f'  特徵: {ALL_FEAT}')

    X_tr  = df_train[ALL_FEAT].values
    y_tr  = df_train['label'].values
    g_tr  = df_train.groupby('uid_hash', sort=False).size().values

    X_val = df_val[ALL_FEAT].values
    y_val = df_val['label'].values
    g_val = df_val.groupby('uid_hash', sort=False).size().values

    ds_train = lgb.Dataset(X_tr,  label=y_tr,  group=g_tr,  feature_name=ALL_FEAT)
    ds_val   = lgb.Dataset(X_val, label=y_val, group=g_val, feature_name=ALL_FEAT,
                           reference=ds_train)

    params = {
        'objective':        'lambdarank',
        'metric':           'ndcg',
        'ndcg_eval_at':     [1, 3, 5],
        'learning_rate':    0.05,
        'num_leaves':       31,
        'min_data_in_leaf': 10,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq':     5,
        'verbosity':        -1,
        'seed':             SEED,
        **LGB_DEVICE_PARAMS,
    }
    print(f'  裝置模式: {"GPU 🟢" if USE_GPU else "CPU 🔵"}')
    print(f'  最大迭代: {N_ROUNDS} rounds，early stopping: {EARLY_STOP} rounds')

    ckpt_path  = OUTPUT_DIR / 'lgbm_checkpoint.txt'
    meta_path  = OUTPUT_DIR / 'train_meta.json'
    init_model = str(ckpt_path) if ckpt_path.exists() else None
    start_iter = 0
    if init_model:
        if meta_path.exists():
            with open(meta_path) as f:
                start_iter = json.load(f).get('n_iter', 0)
        print(f'  發現 checkpoint，從第 {start_iter} 輪繼續...')

    with tqdm(total=start_iter + N_ROUNDS, initial=start_iter,
              desc='  迭代訓練', unit='round') as pbar:
        model = lgb.train(
            params, ds_train,
            num_boost_round=N_ROUNDS,
            valid_sets=[ds_train, ds_val],
            valid_names=['train', 'val'],
            callbacks=[
                lgb.log_evaluation(-1),
                lgb.early_stopping(EARLY_STOP, verbose=False),
                _tqdm_lgb_callback(pbar),
            ],
            init_model=init_model,
        )

    best = model.best_iteration
    print(f'\n  ✓ Early stopping 於第 {best} 輪停止')

    model.save_model(str(ckpt_path))
    with open(OUTPUT_DIR / 'lgbm_ranker.pkl', 'wb') as f:
        pickle.dump(model, f)
    with open(meta_path, 'w') as f:
        json.dump({'n_iter': model.num_trees(), 'best_iteration': best,
                   'feature_names': ALL_FEAT}, f)

    print('\n  Feature importance (gain):')
    pairs = sorted(zip(ALL_FEAT, model.feature_importance('gain')), key=lambda x: -x[1])
    max_s = max(s for _, s in pairs) or 1
    for name, score in pairs:
        bar = '█' * int(score / max_s * 25)
        print(f'    {name:<30} {bar}  {score:.0f}')
    return model


model = train_model(df_train, df_val)

[4/5] 訓練 LightGBM LambdaRank...
  特徵: ['log_uc3', 'log_uc2', 'log_ua', 'log_us', 'log_sc', 'log_sa', 'log_gc', 'log_poi', 'dist']
  裝置模式: GPU 🟢
  最大迭代: 500 rounds，early stopping: 30 rounds


  迭代訓練:   0%|          | 0/500 [00:00<?, ?round/s]


  ✓ Early stopping 於第 1 輪停止

  Feature importance (gain):
    log_uc3                        █████████████████████████  787870
    log_us                           11505
    dist                             3
    log_uc2                          2
    log_gc                           1
    log_ua                           1
    log_sa                           0
    log_sc                           0
    log_poi                          0


## 7｜評估模型

In [16]:
def predict_topk(model: lgb.Booster, query_df: pd.DataFrame,
                  df_sugg: pd.DataFrame, lookups: dict, k: int) -> list[list[str]]:
    """
    對 query_df 每一列輸出 top-k end_latlng。
    候選集 = suggestion 表中該用戶的所有歷史地址（全量，不截斷）。
    新用戶 → fallback 全局最熱門。
    """
    sugg_dict = (df_sugg.groupby('uid_hash')['end_latlng']
                        .apply(lambda g: g.drop_duplicates().tolist())
                        .to_dict())
    global_top = [x for x, _ in
                  sorted(lookups['gc'].items(), key=lambda x: -x[1])[:50]]

    out: list[list[str]] = []
    cols = query_df[['uid_hash','start_latlng','hour_type',
                     'is_holiday','dayofweek']].values

    for uid, start, hour, hol, dow in tqdm(cols, desc='  predict', leave=False):
        cands = sugg_dict.get(uid, [])

        if not cands:
            picks = [x for x in global_top if x != start][:k]
            out.append(picks)
            continue

        feat_rows = [
            _get_feat(lookups, uid, c, hour, hol, dow, start)
            for c in cands
        ]
        scores = model.predict(np.array(feat_rows, dtype=np.float32)[:, :len(ALL_FEAT)])
        ranked = sorted(zip(cands, scores), key=lambda x: -x[1])
        picks  = [c for c, _ in ranked if c != start][:k]

        if len(picks) < k:
            seen = set(picks)
            for x in global_top:
                if x != start and x not in seen:
                    picks.append(x)
                    if len(picks) == k: break

        out.append(picks)
    return out


def run_evaluate(model, query_df, df_sugg, lookups,
                 split_name='test', k_list=TOP_K_LIST):
    """用 evaluate.py 做 row-level 評估。"""
    print(f'[5/5] 評估模型（{split_name} split）...')
    max_k = max(k_list)

    with tqdm(total=2, desc='  評估', unit='step') as pbar:
        pbar.set_postfix_str('推論中')
        predictions = predict_topk(model, query_df, df_sugg, lookups, k=max_k)
        pbar.update(1)
        pbar.set_postfix_str('計算指標')
        truths = query_df['end_latlng'].tolist()
        pbar.update(1)

    print(f'\n  評估筆數: {len(truths):,} 筆')
    print(f'  {"K":<5} {"Hit@K":>10} {"MRR":>10} {"NDCG@K":>10}')
    print(f'  {"-"*38}')

    out = {}
    for k in k_list:
        res = eval_topk(predictions, truths, k=k)
        print(f'  {k:<5} {res.hit_at_k:>10.4f} {res.mrr:>10.4f} {res.ndcg_at_k:>10.4f}')
        out[f'Hit@{k}']  = res.hit_at_k
        out[f'MRR@{k}']  = res.mrr
        out[f'NDCG@{k}'] = res.ndcg_at_k

    with open(OUTPUT_DIR / f'eval_{split_name}.json', 'w') as f:
        import json as _json
        _json.dump(out, f, indent=2)
    print(f'\n  ✓ 評估結果已存至 Drive (eval_{split_name}.json)')
    return predictions, out


val_preds,  val_results  = run_evaluate(model, df_val_raw,  df_sugg, lookups, 'val')
test_preds, test_results = run_evaluate(model, df_test_raw, df_sugg, lookups, 'test')


[5/5] 評估模型（val split）...


  評估:   0%|          | 0/2 [00:00<?, ?step/s]

  predict:   0%|          | 0/100000 [00:00<?, ?it/s]


  評估筆數: 100,000 筆
  K          Hit@K        MRR     NDCG@K
  --------------------------------------
  1         0.3078     0.3078     0.3078
  3         0.5413     0.4086     0.4426
  5         0.6700     0.4379     0.4955

  ✓ 評估結果已存至 Drive (eval_val.json)
[5/5] 評估模型（test split）...


  評估:   0%|          | 0/2 [00:00<?, ?step/s]

  predict:   0%|          | 0/150000 [00:00<?, ?it/s]


  評估筆數: 150,000 筆
  K          Hit@K        MRR     NDCG@K
  --------------------------------------
  1         0.3005     0.3005     0.3005
  3         0.5378     0.4028     0.4374
  5         0.6673     0.4324     0.4907

  ✓ 評估結果已存至 Drive (eval_test.json)


## 8｜推薦示範

In [14]:
# ── 推薦示範 ────────────────────────────────────────────────────────
sample_row = df_test_raw.iloc[0]
uid = sample_row['uid_hash']
recs_raw = predict_topk(model, df_test_raw.iloc[[0]], df_sugg, lookups, k=5)

# end_latlng → end_address 對照
addr_map = df_sugg.drop_duplicates('end_latlng').set_index('end_latlng')['end_address'].to_dict()

print(f'用戶: {uid[:20]}...')
print(f'{"排名":<4} {"下車地址":<40} {"end_latlng"}')
print('-' * 70)
for i, latlng in enumerate(recs_raw[0], 1):
    addr = addr_map.get(latlng, latlng)
    print(f'#{i:<3} {addr:<40} {latlng}')

# ── per-segment 分析（依用戶歷史筆數） ───────────────────────────────
print('\n=== Per-segment 分析（用戶歷史筆數）===')
from evaluate import evaluate_by_segment, user_freq_bucket

truths_test = df_test_raw['end_latlng'].tolist()
seg = user_freq_bucket(df_train_raw, df_test_raw)
seg_df = evaluate_by_segment(test_preds, truths_test, seg, k=5)
print(seg_df.to_string(index=False))
print('\n說明：new(0)=訓練期間沒有行程的新用戶, 1-5=極少歷史, 6-20=中等, 21-100/100+=重度用戶')


  predict:   0%|          | 0/1 [00:00<?, ?it/s]

用戶: c930394ee75d85eb6d66...
排名   下車地址                                     end_latlng
----------------------------------------------------------------------
#1   臺北市信義區基隆路二段149號                          25.027,121.556
#2   台北市萬華區漢口街二段45號12樓                        25.046,121.507
#3   臺北市立聯合醫院忠孝...(臺北市南港區同德路87號)              25.047,121.586
#4   臺北市信義區基隆路一段163號                          25.042,121.565
#5   錢櫃台北忠孝店(臺北市大安區忠孝東路四段22號)                 25.041,121.544

=== Per-segment 分析（用戶歷史筆數）===
segment     n  Hit@5    MRR  NDCG@5
    1-5 55804 0.6662 0.3623  0.4373
 new(0) 42768 0.9239 0.6392  0.7107
   6-20 37074 0.4540 0.3184  0.3518
 21-100 14298 0.4594 0.3840  0.4027
   100+    56 0.1250 0.0848  0.0951

說明：new(0)=訓練期間沒有行程的新用戶, 1-5=極少歷史, 6-20=中等, 21-100/100+=重度用戶


---
## 📥 只想重新評估（已有 checkpoint）
如果 runtime 重啟、模型已存在 Drive，執行以下這格就好，不用重新跑全部。

In [15]:
# ── 從 Drive 載入已訓練模型並重新評估（eval-only 模式）──────────────
with open(OUTPUT_DIR / 'lgbm_ranker.pkl', 'rb') as f:
    model_loaded = pickle.load(f)

# 重新載入資料（若 runtime 重啟）
df_train_raw = load_split('train')
df_val_raw   = load_split('val')
df_test_raw  = load_split('test')

with pq.read_table(DATA_LINK / 'address_v2_suggestion.parquet') as t:
    df_sugg = pd.DataFrame({c: t.column(c).to_pylist() for c in t.column_names})

lookups = build_lookup_tables(df_train_raw)

val_preds,  val_results  = run_evaluate(model_loaded, df_val_raw,  df_sugg, lookups, 'val')
test_preds, test_results = run_evaluate(model_loaded, df_test_raw, df_sugg, lookups, 'test')


TypeError: 'pyarrow.lib.Table' object does not support the context manager protocol